# Tunix-Med: Final Knowledge Evaluation

Evaluates the fine-tuned `google/gemma-3-1b-it` model on questions sampled
**directly from the training dataset** (`lmassaron/medical-cardiology-qa`).

The adapter loaded from `tunix-medical-model/` is the **best checkpoint**
from training — `CleanProgressHook` in notebook 03 overwrites that directory
only when eval loss improves, so the files on disk always reflect the best
step rather than the final step.

1. Reconstructs the held-out split (seed=42, 10% eval) from training.
2. Samples 300 diverse questions from the held-out set.
3. Uses the **same system prompt** the model was trained with.
4. Scores with keyword match, semantic similarity, and an AI judge.

In [1]:
import os, warnings, re
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer, util

warnings.filterwarnings("ignore")


def info_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


device = info_device()
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float32
)

BASE_MODEL = "google/gemma-3-1b-it"
# CleanProgressHook in 03 overwrites this directory every time eval loss
# improves, so this always contains the BEST checkpoint — not the final step.
ADAPTER_PATH = "tunix-medical-model"

print(f"Device : {device}  |  dtype : {dtype}")
print(f"Loading best adapter from : {ADAPTER_PATH}/")

import os, json as _json

cfg_path = os.path.join(ADAPTER_PATH, "adapter_config.json")
if os.path.exists(cfg_path):
    cfg = _json.load(open(cfg_path))
    print(f"  base model     : {cfg.get('base_model_name_or_path')}")
    print(f"  LoRA rank      : {cfg.get('r')}  alpha={cfg.get('lora_alpha')}")
    print(f"  target modules : {cfg.get('target_modules')}")
else:
    print("  ⚠  adapter_config.json not found — run Cell 8 of notebook 03 first.")

print("Loading base model ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype,
    device_map=device,
)
print("Loading LoRA adapter ...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print("Merging adapter into base weights ...")
model = model.merge_and_unload()
model.eval()
print("Model ready.")

Device : cuda  |  dtype : torch.bfloat16
Loading best adapter from : tunix-medical-model/
  base model     : google/gemma-3-1b-it
  LoRA rank      : 16  alpha=32
  target modules : ['gate_proj', 'up_proj', 'down_proj', 'q_proj', 'kv_proj', 'o_proj']
Loading base model ...
Loading LoRA adapter ...
Merging adapter into base weights ...
Model ready.


## Sample Evaluation Questions from the Dataset

We reconstruct the held-out eval split (seed=42, 10% split) and draw 25
questions with broad topic coverage — the same distribution the model trained on.

In [2]:
# ── Load the dataset and reconstruct the eval split ──────────────────────────
# Uses the same seed and split fraction as the training notebook so the
# questions are guaranteed to be from the held-out set the model never trained on.
DATASET_ID = "lmassaron/medical-cardiology-qa"
EVAL_SPLIT = 0.1
SEED = 42
N_EVAL_QS = 300  # number of questions to evaluate

print(f"Loading {DATASET_ID} ...")
full_ds = load_dataset(DATASET_ID, split="train")
n = len(full_ds)

rng = np.random.default_rng(SEED)
all_idx = rng.permutation(n)
cut = int(n * (1.0 - EVAL_SPLIT))
eval_idx = all_idx[cut:]  # same held-out indices as training

print(f"  Total rows  : {n:,}")
print(f"  Eval rows   : {len(eval_idx):,}")


# ── Extract Q-A pairs from the messages format ────────────────────────────────
def extract_qa(example):
    msgs = example["messages"]
    # Format: [system, user, assistant]
    user_msg = next(m["content"] for m in msgs if m["role"] == "user")
    asst_msg = next(m["content"] for m in msgs if m["role"] == "assistant")
    return {"question": user_msg, "answer": asst_msg}


# ── Sample 25 questions with topic diversity ──────────────────────────────────
# Shuffle the eval indices and take the first N_EVAL_QS that have
# non-trivial answers (length > 20 chars) and varied question starts.
rng2 = np.random.default_rng(SEED + 1)
shuffled = rng2.permutation(eval_idx)

seen_prefixes = set()
qa_pairs = []
for idx in shuffled:
    if len(qa_pairs) >= N_EVAL_QS:
        break
    ex = extract_qa(full_ds[int(idx)])
    q, a = ex["question"], ex["answer"]
    # Skip very short answers (likely yes/no)
    if len(a) < 25:
        continue
    # Ensure diversity: track first 4 words of question as a loose topic key
    prefix = " ".join(q.lower().split()[:4])
    if prefix in seen_prefixes:
        continue
    seen_prefixes.add(prefix)
    qa_pairs.append({"question": q, "answer": a, "dataset_idx": int(idx)})

data = pd.DataFrame(qa_pairs)
print(f"\nSampled {len(data)} evaluation questions")
print("\nSample questions:")
for _, row in data.head(5).iterrows():
    print(f"  Q: {row['question'][:90]}")
    print(f"  A: {row['answer'][:80]}")
    print()

Loading lmassaron/medical-cardiology-qa ...
  Total rows  : 10,518
  Eval rows   : 1,052

Sampled 300 evaluation questions

Sample questions:
  Q: What is the rate of bleeding stroke associated with statin use over 5 years of treatment p
  A: Over 5 years of treatment, statins result in 7.5 cases of bleeding stroke per 10

  Q: Which medications used in cardiac stress testing can cause mild hypotension?
  A: Adenosine and dipyridamole.

  Q: What is the advantage of using perfusion stress test with 99mTc labelled sestamibi?
  A: It is appropriate for select patients with an abnormal resting electrocardiogram

  Q: What is the suggested course of action if angina is suspected?
  A: An urgent medical assessment is suggested to rule out serious medical conditions

  Q: What is the benefit of using beta blockers in glaucoma treatment?
  A: They can lower intraocular pressure.



## Inference Loop

In [3]:
# The system prompt used during training — must match exactly
SYSTEM_PROMPT = (
    "You are a knowledgeable medical assistant specializing in cardiology. "
    "Answer clinical questions accurately, focusing on diagnostic criteria, "
    "treatment guidelines, and pathophysiology."
)

results_list = []
model.eval()

for _, row in tqdm(data.iterrows(), total=len(data)):
    question = row["question"]
    answer = row["answer"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    input_ids = encoded.to(device)
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    gen_ids = outputs[0, input_ids.shape[-1] :]
    generated = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    # Reference perplexity (how surprised the model is by the correct answer)
    ref_ids = tokenizer(
        answer, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    full_ids = torch.cat([input_ids, ref_ids], dim=1)
    labels = full_ids.clone()
    labels[:, : input_ids.shape[1]] = -100
    with torch.no_grad():
        loss = model(
            full_ids, attention_mask=torch.ones_like(full_ids), labels=labels
        ).loss
    perplexity = torch.exp(loss).item()

    results_list.append(
        {
            "question": question,
            "expected_answer": answer,
            "generated_answer": generated,
            "perplexity": perplexity,
        }
    )

results_df = pd.DataFrame(results_list)
print(f"Generated {len(results_df)} answers")
results_df[["question", "expected_answer", "generated_answer", "perplexity"]].head()

100%|██████████| 300/300 [01:23<00:00,  3.60it/s]

Generated 300 answers


,question,expected_answer,generated_answer,perplexity
0,What is the rate of bleeding stroke associated...,"Over 5 years of treatment, statins result in 7...","About 27 cases per 10,000 people treated.",2.498536
1,Which medications used in cardiac stress testi...,Adenosine and dipyridamole.,Diuretics and vasodilators.,3.612517
2,What is the advantage of using perfusion stres...,It is appropriate for select patients with an ...,It allows for detection of myocardial ischemia...,27.752718
3,What is the suggested course of action if angi...,An urgent medical assessment is suggested to r...,Immediate assessment for other risk factors an...,12.978132
4,What is the benefit of using beta blockers in ...,They can lower intraocular pressure.,They can help reduce intraocular pressure.,2.658430


## Scoring

In [4]:
# ── Keyword overlap ───────────────────────────────────────────────────────────
def keyword_score(generated, expected):
    kws = set(re.findall(r"\b\w{4,}\b", expected.lower()))
    if not kws:
        return 1.0
    return sum(1 for k in kws if k in generated.lower()) / len(kws)


# ── Semantic similarity ───────────────────────────────────────────────────────
sim_model = SentenceTransformer("all-MiniLM-L6-v2")


def semantic_score(generated, expected):
    e1 = sim_model.encode(generated, convert_to_tensor=True)
    e2 = sim_model.encode(expected, convert_to_tensor=True)
    return float(util.pytorch_cos_sim(e1, e2))


# ── AI judge ─────────────────────────────────────────────────────────────────
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
judge_mdl = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",
)


def ai_judge(question, generated, expected):
    prompt = (
        "You are a medical examiner.\n"
        "Rate the generated answer vs the reference on a 1-10 scale.\n\n"
        f"Question: {question}\n"
        f"Reference: {expected}\n"
        f"Generated: {generated}\n\n"
        "10=perfect, 7-9=mostly correct, 4-6=partial, 1-3=wrong/misleading.\n"
        "Return ONLY the number."
    )
    inp = judge_tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(judge_mdl.device)
    mask = torch.ones_like(inp)
    with torch.no_grad():
        out = judge_mdl.generate(
            inp, attention_mask=mask, max_new_tokens=5, do_sample=False
        )
        txt = judge_tok.decode(
            out[0, inp.shape[-1] :], skip_special_tokens=True
        ).strip()
    m = re.search(r"\d+", txt)
    return int(m.group()) / 10.0 if m else 0.5

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
print("Scoring ...")
results_df["keyword_score"] = results_df.apply(
    lambda r: keyword_score(r["generated_answer"], r["expected_answer"]), axis=1
)
results_df["semantic_score"] = results_df.apply(
    lambda r: semantic_score(r["generated_answer"], r["expected_answer"]), axis=1
)
results_df["ai_judge_score"] = results_df.apply(
    lambda r: ai_judge(r["question"], r["generated_answer"], r["expected_answer"]),
    axis=1,
)

results_df["final_score"] = (
    results_df["keyword_score"] * 0.2
    + results_df["semantic_score"] * 0.4
    + results_df["ai_judge_score"] * 0.4
)

print("\n─── Results ───────────────────────────────────────────────────────")
print(f"  Keyword  score (mean): {results_df['keyword_score'].mean():.3f}")
print(f"  Semantic score (mean): {results_df['semantic_score'].mean():.3f}")
print(f"  AI judge score (mean): {results_df['ai_judge_score'].mean():.3f}")
print(f"  Final    score (mean): {results_df['final_score'].mean():.3f}")
print(f"  Perplexity     (mean): {results_df['perplexity'].mean():.1f}")
print("───────────────────────────────────────────────────────────────────")

# Show best and worst answers
best = results_df.nlargest(3, "final_score")[
    ["question", "expected_answer", "generated_answer", "final_score"]
]
worst = results_df.nsmallest(3, "final_score")[
    ["question", "expected_answer", "generated_answer", "final_score"]
]

print("\n── Top 3 answers ──")
for _, r in best.iterrows():
    print(f"  [{r['final_score']:.2f}]  Q: {r['question'][:70]}")
    print(f"         Expected : {r['expected_answer'][:70]}")
    print(f"         Generated: {r['generated_answer'][:70]}")
    print()

print("── Bottom 3 answers ──")
for _, r in worst.iterrows():
    print(f"  [{r['final_score']:.2f}]  Q: {r['question'][:70]}")
    print(f"         Expected : {r['expected_answer'][:70]}")
    print(f"         Generated: {r['generated_answer'][:70]}")
    print()

Scoring ...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



─── Results ───────────────────────────────────────────────────────
  Keyword  score (mean): 0.220
  Semantic score (mean): 0.535
  AI judge score (mean): 0.609
  Final    score (mean): 0.501
  Perplexity     (mean): 10.9
───────────────────────────────────────────────────────────────────

── Top 3 answers ──
  [0.99]  Q: In which journal can one find information about optimal medical therap
         Expected : New England Journal of Medicine.
         Generated: The New England Journal of Medicine

  [0.96]  Q: What has increased in terms of risk factors for cardiac stress testing
         Expected : The prevalence of diabetes and obesity.
         Generated: The prevalence of obesity and diabetes.

  [0.95]  Q: What are some examples of arrhythmias treated by antiarrhythmic agents
         Expected : Atrial fibrillation, supraventricular tachycardia, and ventricular tac
         Generated: Atrial fibrillation, supraventricular tachycardia, ventricular tachyca

── Bottom 3 answers ──